In [0]:
# EXTRACT — Pull 1 year OHLCV data for 25 Nifty stocks via yfinance API

%pip install yfinance

In [0]:
import pandas as pd
import yfinance as yf
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Data Frames").getOrCreate()

In [0]:
tickers = [
    "TCS.NS", "INFY.NS", "HCLTECH.NS", "WIPRO.NS", "HDFCBANK.NS", 
    "ICICIBANK.NS", "SBIN.NS", "AXISBANK.NS", "BAJFINANCE.NS", "RELIANCE.NS", 
    "NTPC.NS", "ONGC.NS", "POWERGRID.NS", "ITC.NS", "HINDUNILVR.NS", 
    "NESTLEIND.NS", "MARUTI.NS", "M&M.NS", "BAJAJ-AUTO.NS", "TATASTEEL.NS", 
    "HINDALCO.NS", "SUNPHARMA.NS", "CIPLA.NS", "LT.NS", "BHARTIARTL.NS"
]



In [0]:
all_data = []
for ticker in tickers:
    df = yf.download(ticker, period="1y", auto_adjust=True, multi_level_index=False, progress=False)
    df["Ticker"] = ticker
    df = df.reset_index()
    all_data.append(df)

pandas_df = pd.concat(all_data, ignore_index=True)
print(pandas_df.columns.tolist())
print(pandas_df.shape)

In [0]:
spark_df = spark.createDataFrame(pandas_df)
display(spark_df.limit(5))

In [0]:
# VALIDATE — Enforce manual schema and check for nulls

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType

schema = StructType([
    StructField("Date", DateType(), True),
    StructField("Close", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Open", DoubleType(), True),
    StructField("Volume", LongType(), True),
    StructField("Ticker", StringType(), True),
])

spark_df = spark.createDataFrame(pandas_df, schema = schema)
display(spark_df.limit(5))

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

spark_df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in spark_df.columns]).show()

In [0]:
# TRANSFORM — Calculate daily returns, 7-day moving average, sector mapping

from pyspark.sql.functions import col
daily_return = spark_df.withColumn("Daily_Return", ((col("Close") - col("Open"))/col("Open")) * 100 )
display(daily_return.limit(5))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg
from pyspark.sql.functions import col
# 7-day moving average of Close price per ticker.

window_spec = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-6, 0)

In [0]:
# you write this line using avg("Close") and window_spec
df_moving_avg = daily_return.withColumn("Moving_Avg", avg("Close").over(window_spec))
display(df_moving_avg.limit(5))

In [0]:
sector_map = {
    "TCS.NS": "IT",
    "INFY.NS": "IT",
    "HCLTECH.NS": "IT",
    "WIPRO.NS": "IT",
    "RELIANCE.NS": "Energy",
    "ONGC.NS": "Energy",
    "NTPC.NS": "Energy",
    "POWERGRID.NS": "Energy",
    "HDFCBANK.NS": "Banking",
    "ICICIBANK.NS": "Banking",
    "SBIN.NS": "Banking",
    "AXISBANK.NS": "Banking",
    "BAJFINANCE.NS": "Banking",
    "SUNPHARMA.NS": "Pharma",
    "CIPLA.NS": "Pharma",
    "HINDUNILVR.NS": "FMCG",
    "ITC.NS": "FMCG",
    "NESTLEIND.NS": "FMCG",
    "MARUTI.NS": "Auto",
    "BAJAJ-AUTO.NS": "Auto",
    "M&M.NS": "Auto",
    "TATASTEEL.NS": "Metals",
    "HINDALCO.NS": "Metals",
    "LT.NS": "Infrastructure",
    "BHARTIARTL.NS": "Telecom"
}

# Convert dict to list of tuples first
sector_list = [(k, v) for k, v in sector_map.items()]

# Convert to Spark DataFrame
sector_df = spark.createDataFrame(sector_list, ["Ticker", "Sector"])


In [0]:
sector_joined = daily_return.join(sector_df, "Ticker", "inner")
display(sector_joined.limit(5))

In [0]:

from pyspark.sql.functions import col
df_sector = sector_joined.groupBy(col("Sector")).agg(avg(col("Daily_Return")).alias("Avg_Daily_Return"))
display(df_sector.limit(5))

In [0]:
# LOAD — Write partitioned Parquet to ADLS Gen2 + Delta table
 
df_parquet=sector_joined.write.mode("overwrite").partitionBy("Ticker").parquet("abfss://nifty50@alekhyadestorage.dfs.core.windows.net/parquet/")

In [0]:
sector_joined.write.format("delta").mode("overwrite").saveAsTable("nifty50_delta")

In [0]:
# ANALYZE — SparkSQL queries on temp view

# Create temp view on sector_joined
# Query 1: Top 10 stocks by avg daily return
# Query 2: Best performing sector
# Query 3: Top 5 stocks by avg volume

sector_joined.createOrReplaceTempView("nifty50_view")

df_avg_return = spark.sql("SELECT ticker, AVG(daily_return) AS avg_daily_return FROM nifty50_view GROUP BY ticker ORDER BY avg_daily_return DESC LIMIT 10")
display(df_avg_return.limit(5))

# highest avg daily return sector
df_sector = spark.sql("SELECT Sector, AVG(Daily_Return) AS avg_return FROM nifty50_view GROUP BY Sector ORDER BY avg_return DESC LIMIT 1")
display(df_sector)

df_volume = spark.sql("select ticker, avg(volume) as avg_volume from nifty50_view group by ticker order by avg_volume desc limit 5")
display(df_volume)



In [0]:
# TIME TRAVEL — Query previous Delta version
# 
# # Check versions
spark.sql("DESCRIBE HISTORY nifty50_delta").show()

# Query previous version
spark.sql("SELECT * FROM nifty50_delta VERSION AS OF 0 LIMIT 5").show()